# Rotten Fruit Detection Training - Colab

Use this notebook in Google Colab to test the same training code before running it in SageMaker Studio. It expects a zipped dataset with `Train` and `Test` folders.

## 1. Check GPU

In [ ]:
import sys
from pathlib import Path

import tensorflow as tf

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("Working directory:", Path.cwd())

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Clone or Upload Project

If this repo is on GitHub, set `REPO_URL` and run the cell. If you uploaded the project folder manually, set `PROJECT_DIR` to that folder.

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/rotten-fruit-detection-ai.git"
PROJECT_DIR = Path("/content/rotten-fruit-detection-ai")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
print("Project dir:", PROJECT_DIR)
!ls

## 4. Install Dependencies

In [ ]:
%pip install -q -r requirements.txt matplotlib

## 5. Configure Paths

Put your zipped dataset in Google Drive and set `DRIVE_DATA_ZIP_PATH` to it.

In [ ]:
DRIVE_DATA_ZIP_PATH = Path("/content/drive/MyDrive/dataset.zip")
LOCAL_DATA_ZIP_PATH = PROJECT_DIR / "dataset.zip"
EXTRACT_DIR = PROJECT_DIR / "data"
LOCAL_DATA_DIR = EXTRACT_DIR / "dataset"

OUTPUT_DIR = Path("/content/drive/MyDrive/rotten-fruit-artifacts")
MODEL_OUTPUT_PATH = OUTPUT_DIR / "fresh_vs_rotten_model.keras"
CLASS_NAMES_OUTPUT_PATH = OUTPUT_DIR / "fresh_vs_rotten_class_names.txt"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Drive dataset zip:", DRIVE_DATA_ZIP_PATH)
print("Local dataset zip:", LOCAL_DATA_ZIP_PATH)
print("Extract dir:", EXTRACT_DIR)
print("Model output:", MODEL_OUTPUT_PATH)

Expected zipped dataset contents:

```text
dataset/
  Train/
    freshapples/
      image1.jpg
    rottenapples/
      image2.jpg
    freshbanana/
    rottenbanana/
  Test/
    freshapples/
    rottenapples/
    freshbanana/
    rottenbanana/
```

## 6. Prepare Zipped Dataset

In [ ]:
import shutil
import zipfile


def has_train_test(path):
    path = Path(path)
    child_names = {child.name.lower() for child in path.iterdir() if child.is_dir()}
    return "train" in child_names and "test" in child_names


def find_dataset_root(search_dir):
    search_dir = Path(search_dir)
    if has_train_test(search_dir):
        return search_dir

    for path in search_dir.rglob("*"):
        if path.is_dir() and has_train_test(path):
            return path

    raise ValueError(f"Could not find a folder with Train and Test inside {search_dir}.")


if not DRIVE_DATA_ZIP_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset zip: {DRIVE_DATA_ZIP_PATH}")

shutil.copy2(DRIVE_DATA_ZIP_PATH, LOCAL_DATA_ZIP_PATH)

with zipfile.ZipFile(LOCAL_DATA_ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

LOCAL_DATA_DIR = find_dataset_root(EXTRACT_DIR)
print("Training dataset root:", LOCAL_DATA_DIR)

## 7. Load Dataset

In [ ]:
possible_src_dirs = [
    PROJECT_DIR / "src",
    Path.cwd() / "src",
    Path.cwd().parent / "src",
    Path("/content/rotten-fruit-detection-ai/src"),
]

SRC_DIR = next((path for path in possible_src_dirs if path.exists()), None)

if SRC_DIR is None:
    raise FileNotFoundError(
        "Could not find the src folder. Make sure the repo is cloned or uploaded, "
        "then set PROJECT_DIR to the rotten-fruit-detection-ai folder."
    )

sys.path.insert(0, str(SRC_DIR))
print("Using src folder:", SRC_DIR)

from dataset import load_data

train_set, val_set, test_set, class_names = load_data(
    data_dir=LOCAL_DATA_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

print("Classes:", class_names)
print("Number of classes:", len(class_names))

## 8. Build and Train Model

In [ ]:
from model import build_model
from train import train

model = build_model(num_classes=len(class_names))

history = train(
    model=model,
    train_set=train_set,
    val_set=val_set,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

## 9. Plot Training History

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

## 10. Evaluate Test Set

In [ ]:
from train import test_model

test_results = test_model(model, test_set)
test_results

## 11. Save Model and Class Names

In [ ]:
model.save(MODEL_OUTPUT_PATH)
CLASS_NAMES_OUTPUT_PATH.write_text("\n".join(class_names))

print("Saved model to:", MODEL_OUTPUT_PATH)
print("Saved class names to:", CLASS_NAMES_OUTPUT_PATH)

## 12. Test One Image

In [ ]:
import numpy as np
from PIL import Image

TEST_IMAGE_PATH = None

if TEST_IMAGE_PATH is not None:
    image = tf.io.read_file(str(TEST_IMAGE_PATH))
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMAGE_SIZE)
    batch = tf.expand_dims(image, axis=0)

    predictions = model.predict(batch)[0]
    predicted_index = int(np.argmax(predictions))

    display(Image.open(TEST_IMAGE_PATH))
    print("Prediction:", class_names[predicted_index])
    print("Confidence:", float(predictions[predicted_index]))
else:
    print("Set TEST_IMAGE_PATH to test a single image.")